# Phase 0 - Week 2 - Day 2 PM - Database Essentials

## A. Normalization

**Normalization is a database design approach that minimizes data duplication and eliminates unwanted issues such as Insertion, Update, and Deletion Anomalies.** It involves breaking down large tables into smaller ones and establishing relationships between them. The main objective of normalization in SQL is to **remove redundant or repetitive data and ensure data is stored logically.**

There are 7 normal forms that be used for database normalization which are evolve from 1NF to 6NF:
- 1NF (First Normal Form)
- 2NF (Second Normal Form)
- 3NF (Third Normal Form)
- BCNF (Boyce-Codd Normal Form)
- 4NF (Fourth Normal Form)
- 5NF (Fifth Normal Form)
- 6NF (Sixth Normal Form)

However, in most practical, Normalization achieves its best in third normalization.

---

> ***Not Every Database Needs to be Normalized***

While normalization sounds great and awesome and cool, it’s not always the right tool for every situation. In fact, overly normalizing a database that doesn’t need it can lead to complex queries, excessive joins, and slower performance in read-heavy environments.

Normalization is most useful in systems with frequent data changes and a need for accuracy—like OLTP. In read-heavy systems like OLAP, it’s often avoided or only partially applied to improve performance.

| Database Type                | Needs Normalization? | Reason                                |
| ---------------------------- | -------------------- | ------------------------------------- |
| OLTP (transactional)         | Yes              | Data integrity, avoid anomalies (Transactions include insert, update, and delete)       |
| OLAP (analytical/reporting)  | Low (or partially) | Denormalization improves performance  |
| NoSQL or unstructured stores | No      | Schema-less or flexible schema design |


## B. Case Study

In [1]:
# Import library

import pandas as pd
from IPython.core.display import display, HTML

In [2]:
# Data creation

data = {
    "Movie Name": ["The Dark Knight", "Inception", "Parasite"],
    "Genre": ["Action, Crime, Drama", "Action, Sci-Fi", "Drama"],
    "Producer House": ["Warner Bros.", "Syncopy", "Barunson E&A"],
    "Country": ["United States", "United States", "South Korea"],
    "Language": ["English", "English", "Korean"]
}

df_unnormalized = pd.DataFrame(data)
df_unnormalized

,Movie Name,Genre,Producer House,Country,Language
0,The Dark Knight,"Action, Crime, Drama",Warner Bros.,United States,English
1,Inception,"Action, Sci-Fi",Syncopy,United States,English
2,Parasite,Drama,Barunson E&A,South Korea,Korean


### B.1 - 1st Normal Form

The rule of the First Normal Form (1NF) is that values must be atomic, which means:
- Each cell should contain single value
- Each record must be unique (this implies that a primary key is mandatory).

The previous table consist of `Genre` column that have more than single value (e.g. "Action, Crime, Drama" in a single cell). The solution to make the data in 1st normal form is to split `Genre` into single value attribute.

In [3]:
# Create a copy of original data

df_1nf = df_unnormalized.copy()

In [4]:
# Explode multivalued Genre into atomic rows

df_1nf["Genre"] = df_1nf["Genre"].str.split(", ")
df_1nf = df_1nf.explode("Genre").reset_index(drop=True)
df_1nf

,Movie Name,Genre,Producer House,Country,Language
0,The Dark Knight,Action,Warner Bros.,United States,English
1,The Dark Knight,Crime,Warner Bros.,United States,English
2,The Dark Knight,Drama,Warner Bros.,United States,English
3,Inception,Action,Syncopy,United States,English
4,Inception,Sci-Fi,Syncopy,United States,English
5,Parasite,Drama,Barunson E&A,South Korea,Korean


Now that every field has atomic values means we have achieved 1NF.

### B.2 - 2nd Normal Form

The second normal form rules are:
- 1NF rules should be obeyed
- **No partial dependency** (i.e., every non-key attribute must depend on the full primary key)

Since we have achieved 1NF, we need to find wether the table has partial dependency or not. The full primary key (composite key) from the previous table is the combination of `Movie Name` and `Genre` as they make each row unique.

It seems like previous composite key causes partial dependencies in which `Producer House` depends only on `Movie Name` instead of the full primary key.

**Solution:** Split into two tables.

In [5]:
# Movie Table (no repeated rows for `Genre`)
df_movies = df_1nf.drop(columns=["Genre"]).drop_duplicates().reset_index(drop=True)

# Genre Table (`Movie Name` + `Genre` only)
df_genres = df_1nf[["Movie Name", "Genre"]].drop_duplicates().reset_index(drop=True)

In [6]:
# Displaying all the tables in 2NF

display(HTML('<b><h3>Movies Table</h3></b><br>'))
display(df_movies.head())
print('\n')
display(HTML('<b><h3>MovieGenre Table</h3></b><br>'))
display(df_genres.head())

,Movie Name,Producer House,Country,Language
0,The Dark Knight,Warner Bros.,United States,English
1,Inception,Syncopy,United States,English
2,Parasite,Barunson E&A,South Korea,Korean


,Movie Name,Genre
0,The Dark Knight,Action
1,The Dark Knight,Crime
2,The Dark Knight,Drama
3,Inception,Action
4,Inception,Sci-Fi


We created two tables:
- **Movies** with `Movie Name` as the primary key
- **MovieGenre** with the combination of `Movie Name` and `Genre` as composite key

Now, all non-key attributes in each table depend on the full key. We have achieved 2NF.

### B.3 - 3rd Normal Form

The third normal form rules are:
- Should be in 2NF rules
- Has **no transitive dependency** (non-key columns must depend only on the key)

In the previous Movies table let's assume `Producer House` always belongs to one `Country`, this is a **transitive dependency**.

With transitive dependency we just need to separate these into different table.


In [7]:
# `Producer House` Table
df_producers = df_movies[["Producer House", "Country"]].drop_duplicates().reset_index(drop=True)

# Updated Movie Table (no `Country`, only foreign key to `Producer House`)
df_movies_3nf = df_movies.drop(columns=["Country"]).reset_index(drop=True)

In [8]:
# Displaying all the tables in 2nf

display(HTML('<b><h3>Movies Table</h3></b><br>'))
display(df_movies_3nf.head())
print('\n')
display(HTML('<b><h3>MovieGenre Table</h3></b><br>'))
display(df_genres.head())
print('\n')
display(HTML('<b><h3>Producers Table</h3></b><br>'))
display(df_producers.head())

,Movie Name,Producer House,Language
0,The Dark Knight,Warner Bros.,English
1,Inception,Syncopy,English
2,Parasite,Barunson E&A,Korean


,Movie Name,Genre
0,The Dark Knight,Action
1,The Dark Knight,Crime
2,The Dark Knight,Drama
3,Inception,Action
4,Inception,Sci-Fi


,Producer House,Country
0,Warner Bros.,United States
1,Syncopy,United States
2,Barunson E&A,South Korea


The outcome is three tables:
- **Movies** with `Movie Name` as the primary key
- **MovieGenre** with the combination of `Movie Name` and `Genre` as composite key
- **Producer Table** with `Producer House` as the primary key

Now, all non-key columns depend only on the key, and not on other non-key attributes. We have achieved 3NF.